# M3c-T8 Tier 1 throughput probe: Llama-3.1-8B QLoRA (PLAN §7 M3c; D-042, D-055, D-070)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**
(and **GITHUB_TOKEN** only if the repo is private). **Restart the kernel** before *Run all*.

Trains the unharmful and harmful QLoRA endpoints in parallel (one per T4) for 25 steps or 20 min,
on the real WildGuardMix splits, and projects the full run. Tries batch 4×8, then 2×16, then 1×32
until one fits. Expected wall-clock ≈ 30–45 min (Llama download ≈ 16 GB). Probe adapters are
not uploaded; only aggregate JSON is synced. Paste back the output of the last cell.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_m3c_probe.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Download Llama once here, so the two training processes do not race on the same files.
!python -c "from rbbd.models.loading import LoadSpec, download; print(download(LoadSpec('meta-llama/Llama-3.1-8B-Instruct')))"

In [ ]:
# Probe: first batch layout that fits wins (D-070). A non-memory failure stops the loop.
import json, shutil, subprocess, time
CHOSEN = None
for b in (4, 2, 1):
    cfg = f"configs/m3c_probe_llama3.1-8b_b{b}.yaml"
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "rbbd.cli", "run", "--config", cfg, "--stages", "ftdata,train"],
                       capture_output=True, text=True)
    print(f"{cfg}: exit {r.returncode} after {time.time() - t0:.0f} s")
    print(r.stdout[-1200:])
    if r.returncode == 0:
        CHOSEN = b
        break
    print(r.stderr[-3000:])
    if "OutOfMemoryError" not in r.stderr and "out of memory" not in r.stderr:
        break
print("chosen batch layout:", CHOSEN)

In [ ]:
# Copy the probe's aggregate JSON (no adapters, no text) into this session's env folder.
import glob
OUT = f"{ENV_DIR}/m3c_probe"
os.makedirs(OUT, exist_ok=True)
for path in sorted(glob.glob("/kaggle/working/artifacts/train/llama3.1-8b-it/qlora/*/seed0/*/*.json")):
    if path.endswith(("done.json", "data.json")):
        split, key, name = path.split("/")[-4], path.split("/")[-2], path.split("/")[-1]
        shutil.copy(path, f"{OUT}/{split}_{key}_{name}")
for path in sorted(glob.glob("/kaggle/working/artifacts/train/llama3.1-8b-it/qlora/*/seed0/*/train.log")):
    print(path); print("".join(open(path).readlines()[-6:]))

In [ ]:
# Mirror aggregates into the private store (D-041): manifests, ftdata ids/stats, env JSON.
for _path in ["manifests", "ftdata", f"env/{RBBD_SESSION_ID}"]:
    if os.path.isdir(f"/kaggle/working/artifacts/{_path}"):
        !python -m rbbd.cli sync --path {_path}

In [ ]:
# Summary to paste back.
print("chosen batch layout:", CHOSEN)
for path in sorted(glob.glob(f"{OUT}/*.json")):
    d = json.load(open(path)); d.pop("spec", None)
    print(os.path.basename(path)); print(json.dumps(d, indent=1))
!nvidia-smi --query-gpu=index,name,temperature.gpu,clocks.sm --format=csv